# E-Commerce Sales Analysis
## Complete Data Analysis Workflow

This notebook demonstrates a complete data analysis process from loading raw data to extracting business insights.

**Key Questions We'll Answer:**
1. Which products generate the most revenue?
2. What's the revenue breakdown by category?
3. How do sales trends change over time?
4. Who are the most valuable customers?
5. Which regions perform best?
6. What payment methods do customers prefer?

## Section 1: Import Libraries

First, we import all necessary Python libraries:
- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computations
- **matplotlib**: Data visualization (basic plots)
- **seaborn**: Statistical data visualization (beautiful plots)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set visual style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

## Section 2: Load Data

Load the CSV file into a Pandas DataFrame.
A DataFrame is like an Excel spreadsheet - it has rows and columns.

In [ ]:
# Load the data from CSV file
df = pd.read_csv('data/sales_data.csv')

print(f"✅ Data loaded successfully!")
print(f"Total records: {len(df)}")
print(f"Total columns: {len(df.columns)}")

## Section 3: Initial Data Exploration

Let's explore the structure and content of our data

In [ ]:
# Display first 5 rows
print("First 5 rows of data:")
print(df.head())
print("\n" + "="*80 + "\n")

# Display data types
print("Data Types:")
print(df.dtypes)
print("\n" + "="*80 + "\n")

# Display basic statistics
print("Basic Statistics:")
print(df.describe())

## Section 4: Data Quality Check

Check for missing values, duplicates, and other data quality issues

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✅ No missing values found!")
else:
    print(missing[missing > 0])

print("\n" + "="*80 + "\n")

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate Rows: {duplicates}")
if duplicates == 0:
    print("✅ No duplicates found!")

print("\n" + "="*80 + "\n")

# Check data shape
print(f"Data Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

## Section 5: Data Preparation

Convert data types and prepare data for analysis

In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract useful date features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')
df['day_of_week'] = df['date'].dt.day_name()

print("✅ Data preparation complete!")
print("\nNew columns created:")
print("  - year: Extract year from date")
print("  - month: Extract month number")
print("  - month_name: Month as text")
print("  - day_of_week: Day name")

# Display updated data
print("\nUpdated DataFrame:")
print(df.head())

## Section 6: Revenue Analysis by Product

**Question: Which products generate the most revenue?**

Analysis approach:
1. Group by product
2. Sum total revenue
3. Sort from highest to lowest
4. Visualize results

In [ ]:
# Calculate revenue by product
product_revenue = df.groupby('product')['total'].sum().sort_values(ascending=False)

print("Revenue by Product (Top 10):")
print(product_revenue.head(10))
print(f"\nTotal products: {len(product_revenue)}")
print(f"Highest revenue product: {product_revenue.index[0]} (${product_revenue.iloc[0]:,.2f})")
print(f"Lowest revenue product: {product_revenue.index[-1]} (${product_revenue.iloc[-1]:,.2f})")

In [ ]:
# Visualize product revenue
fig, ax = plt.subplots(figsize=(12, 6))

product_revenue.head(10).plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Top 10 Products by Revenue', fontsize=16, fontweight='bold')
ax.set_xlabel('Product', fontsize=12)
ax.set_ylabel('Revenue ($)', fontsize=12)
ax.tick_params(axis='x', rotation=45)

# Add value labels on bars
for i, v in enumerate(product_revenue.head(10)):
    ax.text(i, v + 20, f'${v:,.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("✅ Visualization complete!")

## Section 7: Category Analysis

**Question: What's the revenue breakdown by category?**

Understanding category performance helps identify business strengths.

In [ ]:
# Calculate revenue by category
category_revenue = df.groupby('category').agg({
    'total': 'sum',
    'order_id': 'count',
    'quantity': 'sum'
}).rename(columns={'order_id': 'transaction_count', 'quantity': 'total_units'})

# Calculate percentage
category_revenue['percentage'] = (category_revenue['total'] / category_revenue['total'].sum() * 100).round(2)

print("Revenue by Category:")
print(category_revenue)
print(f"\nTotal Revenue: ${category_revenue['total'].sum():,.2f}")

In [ ]:
# Visualize category revenue with pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart for revenue
colors = ['#FF6B6B', '#4ECDC4']
ax1.pie(category_revenue['total'], labels=category_revenue.index, autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 12})
ax1.set_title('Revenue Distribution by Category', fontsize=14, fontweight='bold')

# Bar chart for transactions
category_revenue['transaction_count'].plot(kind='bar', color=colors, ax=ax2)
ax2.set_title('Transaction Count by Category', fontsize=14, fontweight='bold')
ax2.set_xlabel('Category', fontsize=11)
ax2.set_ylabel('Number of Transactions', fontsize=11)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Category analysis complete!")

## Section 8: Time Series Analysis

**Question: How do sales trends change over time?**

Identifying trends helps with forecasting and planning.

In [ ]:
# Calculate daily revenue
daily_revenue = df.groupby('date')['total'].sum().sort_index()

# Calculate monthly revenue
monthly_revenue = df.groupby('month_name')['total'].sum()

print("Daily Revenue Summary:")
print(f"Average daily revenue: ${daily_revenue.mean():,.2f}")
print(f"Highest daily revenue: ${daily_revenue.max():,.2f} on {daily_revenue.idxmax().date()}")
print(f"Lowest daily revenue: ${daily_revenue.min():,.2f} on {daily_revenue.idxmin().date()}")

print("\n" + "="*80 + "\n")
print("Monthly Revenue:")
print(monthly_revenue)

In [ ]:
# Visualize sales trends
fig, ax = plt.subplots(figsize=(14, 6))

daily_revenue.plot(kind='line', marker='o', color='steelblue', linewidth=2, ax=ax)
ax.set_title('Daily Sales Trend Over Time', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Revenue ($)', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Time series visualization complete!")

## Section 9: Customer Analysis

**Question: Who are the most valuable customers?**

Understanding customer value drives retention strategies.

In [ ]:
# Calculate customer value metrics
customer_stats = df.groupby('customer_id').agg({
    'total': ['sum', 'mean', 'count'],
    'order_id': 'count'
})

# Flatten column names
customer_stats.columns = ['total_spent', 'avg_order_value', 'order_count', 'purchases']
customer_stats = customer_stats.sort_values('total_spent', ascending=False)

print("Top 10 Most Valuable Customers:")
print(customer_stats.head(10))

print("\n" + "="*80 + "\n")
print(f"Total unique customers: {len(customer_stats)}")
print(f"Average customer lifetime value: ${customer_stats['total_spent'].mean():,.2f}")
print(f"Average orders per customer: {customer_stats['order_count'].mean():.2f}")

## Section 10: Regional Performance

**Question: Which regions perform best?**

Regional analysis supports market expansion strategies.

In [ ]:
# Calculate regional performance
region_stats = df.groupby('region').agg({
    'total': 'sum',
    'order_id': 'count',
    'quantity': 'sum'
}).rename(columns={'order_id': 'transactions', 'quantity': 'units_sold'})

region_stats['avg_order_value'] = (region_stats['total'] / region_stats['transactions']).round(2)
region_stats = region_stats.sort_values('total', ascending=False)

print("Regional Performance Summary:")
print(region_stats)
print(f"\nBest performing region: {region_stats.index[0]} (${region_stats['total'].iloc[0]:,.2f})")

In [ ]:
# Visualize regional performance
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# Revenue by region
region_stats['total'].plot(kind='bar', color='steelblue', ax=ax1)
ax1.set_title('Revenue by Region', fontsize=12, fontweight='bold')
ax1.set_ylabel('Revenue ($)', fontsize=10)
ax1.tick_params(axis='x', rotation=45)

# Transactions by region
region_stats['transactions'].plot(kind='bar', color='coral', ax=ax2)
ax2.set_title('Transactions by Region', fontsize=12, fontweight='bold')
ax2.set_ylabel('Count', fontsize=10)
ax2.tick_params(axis='x', rotation=45)

# Average order value by region
region_stats['avg_order_value'].plot(kind='bar', color='lightgreen', ax=ax3)
ax3.set_title('Average Order Value by Region', fontsize=12, fontweight='bold')
ax3.set_ylabel('Average Value ($)', fontsize=10)
ax3.tick_params(axis='x', rotation=45)

# Units sold by region
region_stats['units_sold'].plot(kind='bar', color='plum', ax=ax4)
ax4.set_title('Units Sold by Region', fontsize=12, fontweight='bold')
ax4.set_ylabel('Units', fontsize=10)
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Regional analysis complete!")

## Section 11: Payment Method Analysis

**Question: What payment methods do customers prefer?**

Understanding payment preferences supports payment infrastructure decisions.

In [ ]:
# Calculate payment method statistics
payment_stats = df.groupby('payment_method').agg({
    'total': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'transaction_count'})

payment_stats['percentage'] = (payment_stats['total'] / payment_stats['total'].sum() * 100).round(2)
payment_stats['avg_transaction'] = (payment_stats['total'] / payment_stats['transaction_count']).round(2)
payment_stats = payment_stats.sort_values('total', ascending=False)

print("Payment Method Analysis:")
print(payment_stats)
print(f"\nMost popular payment method: {payment_stats.index[0]}")

In [ ]:
# Visualize payment methods
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Revenue distribution
colors_payment = ['#FF6B6B', '#4ECDC4', '#45B7D1']
payment_stats['total'].plot(kind='bar', color=colors_payment, ax=ax1)
ax1.set_title('Revenue by Payment Method', fontsize=14, fontweight='bold')
ax1.set_xlabel('Payment Method', fontsize=11)
ax1.set_ylabel('Revenue ($)', fontsize=11)
ax1.tick_params(axis='x', rotation=45)

# Transaction count
payment_stats['transaction_count'].plot(kind='pie', autopct='%1.1f%%', colors=colors_payment, ax=ax2)
ax2.set_title('Transaction Distribution', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

print("✅ Payment analysis complete!")

## Section 12: Executive Summary & Key Findings

Here's a comprehensive summary of our analysis.

In [ ]:
print("""\n
{'='*80}
EXECUTIVE SUMMARY - E-COMMERCE SALES ANALYSIS
{'='*80}\n""")

print("📊 OVERALL METRICS")
print("-" * 80)
print(f"Total Revenue: ${df['total'].sum():,.2f}")
print(f"Total Transactions: {len(df)}")
print(f"Average Order Value: ${df['total'].mean():,.2f}")
print(f"Unique Customers: {df['customer_id'].nunique()}")
print(f"Date Range: {df['date'].min().date()} to {df['date'].max().date()}")

print("\n📈 TOP PRODUCTS")
print("-" * 80)
for i, (product, revenue) in enumerate(product_revenue.head(5).items(), 1):
    pct = (revenue / df['total'].sum() * 100)
    print(f"{i}. {product:20} | ${revenue:>10,.2f} | {pct:>5.1f}%")

print("\n🏆 TOP CUSTOMERS")
print("-" * 80)
for i, (customer, row) in enumerate(customer_stats.head(5).iterrows(), 1):
    print(f"{i}. {customer:10} | ${row['total_spent']:>10,.2f} | {int(row['order_count'])} orders")

print("\n🗺️ REGIONAL PERFORMANCE")
print("-" * 80)
for region in region_stats.index:
    revenue = region_stats.loc[region, 'total']
    transactions = region_stats.loc[region, 'transactions']
    pct = (revenue / df['total'].sum() * 100)
    print(f"{region:10} | ${revenue:>10,.2f} | {transactions:>3} trans | {pct:>5.1f}%")

print("\n💳 PAYMENT METHODS")
print("-" * 80)
for method in payment_stats.index:
    revenue = payment_stats.loc[method, 'total']
    trans = payment_stats.loc[method, 'transaction_count']
    pct = (revenue / df['total'].sum() * 100)
    print(f"{method:15} | ${revenue:>10,.2f} | {trans:>3} trans | {pct:>5.1f}%")

print("\n✅ KEY INSIGHTS")
print("-" * 80)
print(f"1. {product_revenue.index[0]} is the top revenue generator")
print(f"2. {region_stats.index[0]} region shows strongest performance")
print(f"3. {payment_stats.index[0]} is the most popular payment method")
print(f"4. Average customer lifetime value: ${customer_stats['total_spent'].mean():,.2f}")
print(f"5. {category_revenue.index[0]} category dominates with {category_revenue.loc[category_revenue.index[0], 'percentage']:.1f}% of revenue")

print("\n💡 RECOMMENDATIONS")
print("-" * 80)
print("1. Focus marketing on high-performing regions")
print("2. Expand best-selling product lines")
print("3. Develop loyalty program for top customers")
print("4. Optimize payment processing for preferred methods")
print("5. Investigate seasonal trends for better forecasting")

print("\n" + "="*80 + "\n")